# Topic Modeling of Recent Academic Publications using CombinedTM
## Discovering Hot Research Topics from 2020 to 2025

In [ ]:
!pip install -qq numpy==1.26.4 gensim
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
!git clone https://github.com/MilaNLProc/contextualized-topic-models.git
%cd contextualized-topic-models
!pip install -e .

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. Dataset

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/MachineLearning/processed.csv")
print(df.shape)
df.head()

In [ ]:
df = df.dropna(subset=['cleaned_abstract'])
documents = df['cleaned_abstract'].tolist()
years = df["year"].astype(str).tolist()
print(f"Total documents: {len(documents)}")
print("First document sample:")
print(documents[0][:300], "...")

## 2. Modeling

### 2.1. Preprocessing

In [ ]:
from contextualized_topic_models.models.ctm import CombinedTM
from contextualized_topic_models.utils.data_preparation import TopicModelDataPreparation
from contextualized_topic_models.utils.preprocessing import WhiteSpacePreprocessingStopwords
import nltk
from nltk.corpus import stopwords as stop_words

nltk.download('stopwords')

stopwords = list(stop_words.words("english"))

sp = WhiteSpacePreprocessingStopwords(documents, stopwords_list=stopwords)
preprocessed_documents, unpreprocessed_corpus, vocab, retained_indices = sp.preprocess()

In [ ]:
preprocessed_documents[0]

### 2.2. Embedding and Contextual + BoW Inputs

In [ ]:
tp = TopicModelDataPreparation("all-mpnet-base-v2")

training_dataset = tp.fit(text_for_contextual=unpreprocessed_corpus, text_for_bow=preprocessed_documents)

### 2.3. CombinedTM Model

#### Training

In [ ]:
ctm = CombinedTM(bow_size=len(tp.vocab),
                 contextual_size=768,
                 n_components=20,
                 num_epochs=20)
ctm.fit(training_dataset, verbose=True)

In [ ]:
topic_word_matrix = ctm.get_topic_word_distribution()
num_topics = topic_word_matrix.shape[0]
print(f"Topics found: {num_topics}")

In [ ]:
ctm.get_topic_lists(5)

#### Inference

In [ ]:
import time

start_time = time.time()

predicted_topics = ctm.get_predicted_topics(training_dataset, n_samples=20)

end_time = time.time()

inference_time = end_time - start_time
print(f"inference süresi: {inference_time:.2f} saniye")

In [ ]:
preprocessed_documents[0]

### 3. Visualizations

In [ ]:
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics.pairwise import cosine_distances
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt

topic_vecs = ctm.get_topic_word_distribution()

topic_dict = ctm.get_topics(k=10)
labels = [
    f"{tid} | {', '.join(words[:3])}" for tid, words in topic_dict.items() if isinstance(words, list)
]

dist_matrix = cosine_distances(topic_vecs)

condensed_dist = squareform(dist_matrix, checks=False)
linkage_matrix = linkage(condensed_dist, method='average')

plt.figure(figsize=(12, 20))
dendrogram(
    linkage_matrix,
    orientation='right',
    labels=labels,
    leaf_font_size=12,
)
plt.title("CTM Topic Hierarchy")
plt.xlabel("Distance")
plt.ylabel("Topics")
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import seaborn as sns
import matplotlib.pyplot as plt

topic_vecs = ctm.get_topic_word_distribution()

sim_matrix = cosine_similarity(topic_vecs)

plt.figure(figsize=(10, 8))
sns.heatmap(
    sim_matrix,
    xticklabels=False,
    yticklabels=False,
    cmap='coolwarm',
    square=True,
    cbar=True
)

plt.title("CTM Topic-Topic Cosine Similarity Heatmap", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import umap
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

thetas = ctm.get_thetas(training_dataset)

umap_model = umap.UMAP(n_components=2, random_state=42)
doc_vecs_2d = umap_model.fit_transform(thetas)

dominant_topic_ids = np.argmax(thetas, axis=1)

plt.figure(figsize=(12, 8))
sns.scatterplot(
    x=doc_vecs_2d[:, 0],
    y=doc_vecs_2d[:, 1],
    hue=dominant_topic_ids,
    palette="tab10",
    legend=False,
    s=10,
    alpha=0.7
)

plt.title("CTM Document Clusters via UMAP")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.tight_layout()
plt.show()

In [ ]:
topicdf = df.iloc[retained_indices].reset_index(drop=True)
topicdf["predicted_topic"] = predicted_topics

In [ ]:
topics_words = ctm.get_topics(k=10)
topicdf["predicted_topic_words"] = topicdf["predicted_topic"].apply(lambda x: ", ".join(topics_words[x]))

In [ ]:
topicdf[["cleaned_abstract", "predicted_topic", "predicted_topic_words"]]

In [ ]:
topic_dict = ctm.get_topics(k=10)
topics = [words for words in topic_dict.values() if isinstance(words, list)]

topics_per_doc = topicdf["predicted_topic"]
years = topicdf["year"]

df_topic_time = pd.DataFrame({
    "Year": years,
    "Topic": topics_per_doc
})

topic_counts = df_topic_time.groupby(["Year", "Topic"]).size().reset_index(name="Frequency")

pivot = topic_counts.pivot(index="Year", columns="Topic", values="Frequency").fillna(0)

first_year = pivot.index.min()
last_year = pivot.index.max()
growth = pivot.loc[last_year] - pivot.loc[first_year]
top_growing = growth.sort_values(ascending=False).head(10).index.tolist()

### Topics over Time - Growing Topics from 2020 to 2025

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))

for tid in top_growing:
    keywords = topic_dict.get(tid, [])[:5]
    label = f"Topic {tid}: {', '.join(keywords)}"
    plt.plot(pivot.index, pivot[tid], marker='o', label=label)

plt.title("En Hızlı Büyüyen Araştırma Konuları", fontsize=14)
plt.xlabel("Year")
plt.ylabel("Document Count")
plt.legend(fontsize=9)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
for topic_id in top_growing:
    if topic_id in topic_dict:
        keywords = topic_dict[topic_id]
        print(f"Topic {topic_id}: {', '.join(keywords)}")

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

growing_topics = [0, 8, 5]

for topic_id in growing_topics:
    words = topic_dict.get(topic_id, [])
    freqs = {word: 1 for word in words}

    wc = WordCloud(width=600, height=400, background_color='white').generate_from_frequencies(freqs)

    print(f"Topic {topic_id}")
    plt.figure(figsize=(6, 4))
    plt.imshow(wc, interpolation='bilinear')
    plt.axis('off')
    plt.title(f"Topic {topic_id}", fontsize=14)
    plt.tight_layout()
    plt.show()

#### Growing Topics from 2020 to 2025

| Topic ID | Topic Title                                      |
|----------|--------------------------------------------------|
| 0        | Multimodal LLM Reasoning                         |
| 8        | Efficient Inference & Model Optimization         |
| 5        | Diffusion-Based Image & Video Generation         |
| 3        | LLMs for Knowledge Retrieval & QA                |
| 2        | Social Impact of AI in Education & Media         |
| 14       | Robotics & Autonomous Vehicle Control            |
| 13       | Federated Learning & Privacy-Preserving ML       |
| 7        | 3D Scene Understanding & Pose Estimation         |
| 17       | Visual Semantic Segmentation & Fusion            |
| 18       | Reinforcement Learning & Policy Optimization     |


## 4. Performance Metrics

### 4.1. Coherence Scores

In [ ]:
from gensim.corpora import Dictionary
from gensim.models.coherencemodel import CoherenceModel

tokenized_docs = [doc.split() for doc in topicdf["cleaned_abstract"]]
dictionary = Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

topics = [words for words in topic_dict.values() if isinstance(words, list)]

coherence_cv = CoherenceModel(topics=topics, texts=tokenized_docs, dictionary=dictionary, coherence='c_v').get_coherence()
coherence_umass = CoherenceModel(topics=topics, texts=tokenized_docs, dictionary=dictionary, coherence='u_mass').get_coherence()
coherence_npmi = CoherenceModel(topics=topics, texts=tokenized_docs, dictionary=dictionary, coherence='c_npmi').get_coherence()
coherence_uci = CoherenceModel(topics=topics, texts=tokenized_docs, dictionary=dictionary, coherence='c_uci').get_coherence()

print(f"C_v Coherence:     {coherence_cv:.4f}")
print(f"U_Mass Coherence: {coherence_umass:.4f}")
print(f"NPMI Coherence:   {coherence_npmi:.4f}")
print(f"UCI Coherence:    {coherence_uci:.4f}")

### 4.2. PUW

In [ ]:
all_words = [word for topic in topics for word in topic]

unique_words = set(all_words)
puw = len(unique_words) / len(all_words)

print(f"PUW: {puw:.4f}")

### 4.3. Avg. Jaccard Similarity

In [ ]:
from itertools import combinations

def jaccard_similarity(set1, set2):
    return len(set1 & set2) / len(set1 | set2)

jaccard_scores = []
for t1, t2 in combinations(topics, 2):
    jaccard_scores.append(jaccard_similarity(set(t1), set(t2)))

avg_jaccard = sum(jaccard_scores) / len(jaccard_scores)
print(f"Jaccard: {avg_jaccard:.4f}")